In [7]:
import pandas as pd
import numpy as np
import glob
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error

# Reaggregating the data
labels = pd.read_parquet('../fastf1_data/labeled/labels_combined.parquet')
segmented_files = glob.glob('../fastf1_data/processed/corners_*.parquet')
segmented = pd.concat([pd.read_parquet(f) for f in segmented_files], ignore_index=True)

target_cols = ['aggression_score', 'line_shape_score', 'oversteer_preference_score']
SAFE_COLUMNS = ['Throttle', 'Brake', 'Speed', 'nGear', 'RPM']  

# Find the length to truncate corners to
lengths = segmented.groupby(['driver', 'lap_number', 'corner_number']).size()
# print(lengths.describe())

lengths_sorted = lengths.sort_values(ascending=False)
# print(lengths_sorted.head(10))

MAX_SEQ_LEN = 100   # Covers most corners (75th percentile = 85); longer sequences are
                    # truncated to their first MAX_SEQ_LEN samples. A small number of
                    # corners (especially corner 1, sometimes 18) have anomalously long
                    # sequences — likely a slicing bug in corner_slicer.py (see notes),
                    # not real corner telemetry. Truncating avoids letting them dominate
                    # padding/memory without excluding them.

sequences, targets, groups = [], [], []

for (driver, lap_num, corner_num), group in segmented.groupby(['driver', 'lap_number', 'corner_number']):
    match = labels[
        (labels.driver == driver) &
        (labels.lap_number == lap_num) &
        (labels.corner_number == corner_num)
    ]
    if match.empty:
        continue  # Corner instance didn't survive the out-lap filter in labeling

    seq = group.sort_values('Distance')[SAFE_COLUMNS].to_numpy(dtype=np.float64)

    if len(seq) >= MAX_SEQ_LEN:
        seq = seq[:MAX_SEQ_LEN]
    else:
        pad = np.zeros((MAX_SEQ_LEN - len(seq), len(SAFE_COLUMNS)))
        seq = np.vstack([seq, pad])

    sequences.append(seq)
    targets.append(match[target_cols].values[0])
    groups.append(driver)

X = np.array(sequences)   # hape: (n_corners, MAX_SEQ_LEN, n_features)
y = np.array(targets)     # shape: (n_corners, 3)
groups = np.array(groups)

print(f"X shape: {X.shape}, y shape: {y.shape}")


X shape: (19740, 100, 5), y shape: (19740, 3)


In [8]:
from sklearn.model_selection import GroupShuffleSplit

# Split by DRIVER (not randomly) — prevents the model from
# seeing the same driver's data in both train and validation
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(splitter.split(X, groups=groups))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

# Scale features (fit on train only)
# X_train shape is (n_corners, seq_len, n_features) — flatten across corners+time
# to compute per-channel mean/std, then reshape back
n_train, seq_len, n_feat = X_train.shape
flat_train = X_train.reshape(-1, n_feat)

mean = flat_train.mean(axis=0)
std = flat_train.std(axis=0)
std[std == 0] = 1e-8  # avoid divide-by-zero on any constant channel

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std

Train: (15293, 100, 5), Val: (4447, 100, 5)


In [ ]:
class CornerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(CornerDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(CornerDataset(X_val, y_val), batch_size=32)


class LSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden_size=32, n_outputs=3):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_outputs)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)   # h_n: final hidden state after the full sequence
        return self.fc(h_n[-1])


model = LSTMRegressor(n_features=len(SAFE_COLUMNS))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

n_epochs = 50

for epoch in range(n_epochs):
    model.train()
    total_train_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item() * xb.size(0)
    avg_train_loss = total_train_loss / len(train_loader.dataset)

    # quick validation pass each epoch, no gradient tracking needed
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            preds = model(xb)
            loss = criterion(preds, yb)
            total_val_loss += loss.item() * xb.size(0)
    avg_val_loss = total_val_loss / len(val_loader.dataset)
    print(f"Epoch {epoch+1}/{n_epochs} — train MSE: {avg_train_loss:.4f}, val MSE: {avg_val_loss:.4f}")
    
model.eval()
with torch.no_grad():
    val_preds = model(torch.tensor(X_val, dtype=torch.float32)).numpy()

mae = mean_absolute_error(y_val, val_preds)
print(f"LSTM MAE: {mae:.4f}")
for i, col in enumerate(target_cols):
    col_mae = mean_absolute_error(y_val[:, i], val_preds[:, i])
    print(f"{col} MAE: {col_mae:.4f}")

torch.save(model.state_dict(), '../models/lstm_model.pt')

Epoch 1/50 — train MSE: 0.0978, val MSE: 0.0715
Epoch 2/50 — train MSE: 0.0704, val MSE: 0.0562
Epoch 3/50 — train MSE: 0.0383, val MSE: 0.0302
Epoch 4/50 — train MSE: 0.0295, val MSE: 0.0254
Epoch 5/50 — train MSE: 0.0263, val MSE: 0.0257
Epoch 6/50 — train MSE: 0.0255, val MSE: 0.0252
Epoch 7/50 — train MSE: 0.0240, val MSE: 0.0225
Epoch 8/50 — train MSE: 0.0219, val MSE: 0.0202
Epoch 9/50 — train MSE: 0.0236, val MSE: 0.0198
Epoch 10/50 — train MSE: 0.0207, val MSE: 0.0198
Epoch 11/50 — train MSE: 0.0198, val MSE: 0.0184
Epoch 12/50 — train MSE: 0.0199, val MSE: 0.0195
Epoch 13/50 — train MSE: 0.0187, val MSE: 0.0181
Epoch 14/50 — train MSE: 0.0195, val MSE: 0.0200
Epoch 15/50 — train MSE: 0.0190, val MSE: 0.0181
Epoch 16/50 — train MSE: 0.0178, val MSE: 0.0182
Epoch 17/50 — train MSE: 0.0175, val MSE: 0.0169
Epoch 18/50 — train MSE: 0.0164, val MSE: 0.0166
Epoch 19/50 — train MSE: 0.0163, val MSE: 0.0161
Epoch 20/50 — train MSE: 0.0169, val MSE: 0.0188
Epoch 21/50 — train MSE: 0.01